In [1]:
# ========================== CONFIG ==========================
import os
import re
from datetime import date, timedelta, datetime
from zoneinfo import ZoneInfo

# ---- Date window (start anywhere, include N days)
START_DATE   = date(2026, 6, 1)
NUM_DAYS     = 7
DAY_DATES    = [START_DATE + timedelta(days=i) for i in range(NUM_DAYS)]
# Column labels (change to "%a %m/%d" if you prefer dates in headers)

def _col_label(d: date) -> str:
    try:
        return d.strftime("%A %-m/%-d")     # Mac/Linux
    except ValueError:
        return d.strftime("%A %#m/%#d")     # Windows

COL_LABELS = [_col_label(d) for d in DAY_DATES]


# ---- Output -----#
# --- Specify output directory and filename - creates directory if needed
OUTPUT_DIR   = "output"
OUTPUT_XLSX  = f"StoryPointEL_Listings_{START_DATE}_{NUM_DAYS}_AYWT_SMW_3-1.xlsx"
os.makedirs(OUTPUT_DIR, exist_ok=True)



# ---- Channel-number profile (sports-relevant only)
CHANNEL_MAPS = {
    "StoryPoint EL": {
        # Locals
        "CBS": 3, "NBC": 4, "FOX": 6, "ABC": 7,
        # Sports tier
        "ESPN": 11, "ESPNews": 12, "ESPNU": 13, "ESPN2": 14,
        "FS1": 15, "Tigers TV": 48,
        # Golf scoreboard feeds are handled with the same ESPN scoreboard parser now,
        # but exact round-by-round TV windows may still need a supplemental guide source.
        "Golf": 52,
        # Entertainment nets that carry sports
        "USA": 18, "TNT": 19, "truTV": 20, "TBS": 21,
        # Streaming / other
        "B1G+ APP": 98, "Peacock": 99,
    }
}
ACTIVE_CHANNEL_MAP_NAME = "StoryPoint EL"

# ---- ESPN scoreboard URL overrides
# Most feeds work from site.api.espn.com using the league key directly.
# College softball is an exception discovered in 2025/2026: ESPN exposes it
# under baseball/college-softball on the site.web.api host.
SCOREBOARD_URL_OVERRIDES = {
    "baseball/college-softball": "https://site.web.api.espn.com/apis/site/v2/sports/baseball/college-softball/scoreboard",
}


# ================== SCOREBOARD CATALOG (comment to disable) ==================
SPORTS = [
    ("NFL", ["football/nfl"]),
    ("NCAA FB", ["football/college-football"]),
    ("NBA", ["basketball/nba"]),
    ("NHL", ["hockey/nhl"]),
    ("NCAA Hockey", ["hockey/mens-college-hockey"]),
    # ("NCAA Hockey (M)", ["hockey/mens-college-hockey"]),
    ("MLB", ["baseball/mlb"]),
    
    ("NCAA BB (M)", ["basketball/mens-college-basketball"]),
    ("NCAA BB (W)", ["basketball/womens-college-basketball"]),
    
    ### All Known Available Leagues (uncomment to enable more)
    ("NCAA Baseball", ["baseball/college-baseball"]),
    # ("NCAA Lacrosse(M)", ["lacrosse/mens-college-lacrosse"]),
    ("NCAA Soccer(M)", ["soccer/usa.ncaa.m.1"]),
    # ("Volleyball(M)", ["volleyball/mens-college-volleyball"]),
    # ("NCAA Water Polo(M)", ["waterpolo/mens-college-water-polo"]),
    ("Volleyball", ["volleyball/womens-college-volleyball"]),
    ("NCAA Softball", ["baseball/college-softball"]),
    # ("NCAA Field Hockey", ["fieldhockey/womens-college-field-hockey"]), # # Field Hockey
    # ("NCAA Ice Hockey(W)", ["hockey/womens-college-hockey"]), # # Womens Ice Hockey
    # ("NCAA Lacrosse(W)", ["lacrosse/womens-college-lacrosse"]), # # Lacrosse (W)
    ("NCAA Soccer(W)", ["soccer/usa.ncaa.w.1"]), # # Soccer Women

    # Golf scoreboard feeds return tournament-level events. They usually include
    # networks, but not always exact daily broadcast windows.
    ("PGA", ["golf/pga"]),
    ("LPGA", ["golf/lpga"]),
    ("Champions Tour", ["golf/champions-tour"]),
    # ("LIV Golf", ["golf/liv"]),
    # ("DP World Tour", ["golf/eur"]),

    # ("NCAA Volleyball(W)", ["volleyball/womens-college-volleyball"]), # # Volleyball (W)
    # ("NCAA Water Polo(W)", ["waterpolo/womens-college-water-polo "]), # # Water Polo (W)

    # # Soccer
    ("MLS",   ["soccer/usa.1"]), # MLS USA
    ("EPL",   ["soccer/eng.1"]), # ENGLISH PREMIER
    ("UCL",   ["soccer/uefa.champions"]), # UEFA CHAMPIONS LEAGUE
    # ("La Liga", ["soccer/esp.1"]), # # La Liga
    # ("Bundesliga", ["soccer/ger.1"]), # # Bundesliga
    # ("NWSL",  ["soccer/usa.nwsl"]), # # National Womens NWSL (USA)
    
    # ("Liga MX", ["soccer/mex.1"]), # # Liga MX (Mexico)
]

# ================== TV GUIDE BACKUP / SUPPLEMENT ==================
# ESPN scoreboard feeds are still the main source for games. This optional
# guide layer supplements the scoreboard with sports-only station listings,
# which helps with featured windows and especially cleaner Golf Channel rows.
USE_TV_GUIDE_BACKUP = True
TV_GUIDE_SOURCE = "areyouwatchingthis"   # sports-only station schedules
TV_GUIDE_FILL_MODE = "append_dedupe"     # append guide rows, then drop duplicate channel/date/time/tag rows
TV_GUIDE_DEBUG_OUTPUTS = True            # always write parsed/kept/rejected guide rows to CSV
TV_GUIDE_EVENT_STRICTNESS = "live_only"  # "live_only" or "featured_sports"

# AYWT station pages include useful live/featured events, but they also include
# overnight replay blocks. These settings keep the guide layer focused on
# "live-looking" broadcast windows instead of filling the printable grid with
# 3:00am/4:00am reruns.
TV_GUIDE_DROP_EARLY_MORNING_REPLAYS = True
TV_GUIDE_EARLY_MORNING_CUTOFF_HOUR = 6  # drop guide rows before 6:00am unless you explicitly loosen this later
TV_GUIDE_DEDUPLICATE_SAME_TITLE_BY_DAY = True  # keep the first airing per channel/date/tag/title
TV_GUIDE_STRIP_VENUE_FROM_TITLES = True        # remove trailing "at Ball Arena" / "at Historic Crew Stadium" bulk
TV_GUIDE_DROP_GUIDE_ROWS_NEAR_SCOREBOARD = True  # scoreboard wins if a guide row is the same channel/tag near the same time
TV_GUIDE_SCOREBOARD_OVERLAP_MINUTES = 20

# For normal channel rows, use guide listings instead of ESPN scoreboard tournament feeds for golf.
# The scoreboard golf items can be useful, but they tend to create all-day/generic rows.
USE_GOLF_SCOREBOARD_FEEDS = False
GOLF_SCOREBOARD_LEAGUES = {"golf/pga", "golf/lpga", "golf/champions-tour", "golf/liv", "golf/eur"}

# Are You Watching This?! station pages are sports-focused and usually expose
# about two weeks of sports programming in simple text form.
# Add/remove channels here without touching the parser.
TV_GUIDE_CHANNEL_URLS = {
    "ESPN": "https://areyouwatchingthis.com/tv/stations/espn",
    "ESPN2": "https://areyouwatchingthis.com/tv/stations/espn2",
    "ESPNU": "https://areyouwatchingthis.com/tv/stations/espnuhd-espnu-hd",
    "ESPNews": "https://areyouwatchingthis.com/tv/stations/espnews",
    "Golf": "https://areyouwatchingthis.com/tv/stations/golf-the-golf-channel",
    "FS1": "https://areyouwatchingthis.com/tv/stations/fs1-fox-sports-1",
    "TNT": "https://areyouwatchingthis.com/tv/stations/tntphd-turner-network-tv-hd-pacific",
    "TBS": "https://areyouwatchingthis.com/tv/stations/tbshd-tbs-hd",
    "USA": "https://areyouwatchingthis.com/tv/stations/usa-network",
    # Lansing-area local affiliates. These are helpful for general sports backups,
    # but AYWT can still miss golf on broadcast networks, which is why the
    # Sports Media Watch golf schedule layer below exists.
    "NBC": "https://areyouwatchingthis.com/tv/stations/wilxdt-nbc",
    "CBS": "https://areyouwatchingthis.com/tv/stations/wlnsdt-cbs",
    "FOX": "https://areyouwatchingthis.com/tv/stations/wsymdt-fox",
    "ABC": "https://areyouwatchingthis.com/tv/stations/wlajdt-abc",
    # These station pages may exist but may be sparse depending on provider mapping.
    # Leave them enabled; empty pages simply produce zero guide rows.
    # "truTV": "https://areyouwatchingthis.com/tv/stations/trutvhd-trutv-hd",
}

# Keep the guide scrape focused on event telecasts rather than studio shows,
# documentaries, betting shows, and highlights.

# ================== GOLF-SPECIFIC SCHEDULE SUPPLEMENT ==================
# AYWT is excellent for Golf Channel rows, but its USA/NBC/CBS/ABC affiliate
# pages can miss network golf windows. Sports Media Watch publishes golf-specific
# TV schedule pages with exact network windows, including NBC/USA/Peacock blocks
# such as the U.S. Women's Open.
USE_GOLF_SCHEDULE_BACKUP = True
GOLF_SCHEDULE_SOURCE = "sportsmediawatch"
GOLF_SCHEDULE_DEBUG_OUTPUTS = True

# Keep Golf Channel on AYWT by default because that source has been cleaner in
# your local tests. The SMW layer is mainly to fill network/streaming golf windows.
GOLF_SCHEDULE_SKIP_CHANNELS = {"Golf"}

GOLF_SCHEDULE_URLS = {
    "LPGA": "https://www.sportsmediawatch.com/tv-schedules/lpga-tv-schedule/",
    "PGA Tour": "https://www.sportsmediawatch.com/tv-schedules/pga-tour-tv-schedule/",
    "LIV Golf": "https://www.sportsmediawatch.com/tv-schedules/liv-golf-tv-schedule/",
    # Event-specific pages can be added here when a major has better coverage
    # than the tour-level page. The de-dupe layer will keep only one copy.
    "US Women's Open": "https://www.sportsmediawatch.com/tv-schedules/lpga-tv-schedule/us-womens-open/",
}

# Sports Media Watch can include international feeds after a pipe, e.g.
# "CBS, Paramount+ | CTV2, TSN4". The notebook is for the U.S. channel grid, so
# keep the left side by default.
GOLF_SCHEDULE_US_ONLY = True

# Avoid using the golf-specific schedule to add early featured groups unless the
# channel is actually in your lineup. ESPN+ is already excluded unless you add it
# to CHANNEL_MAPS.
GOLF_SCHEDULE_EXCLUDE_CHANNEL_KEYS = {
    "espn+", "espn plus", "espn select", "espn unlimited",
    "paramount+", "paramount plus",
    "youtube tv", "directv", "xfinity", "usga app", "uswomensopen.com",
    "tsn", "tsn4", "tsn+", "ctv2", "siriusxm", "radio", "audio",
}

TV_GUIDE_EXCLUDE_TITLE_RE = re.compile(
    r"\b("
    r"SportsCenter|SportCenter|Get Up|First Take|Pardon the Interruption|PTI|Around the Horn|"
    r"NBA Today|NFL Live|MLB Tonight|College Football Live|College GameDay|SEC Now|"
    r"Golf Central|Golf Today|The Golf Fix|Ask Rory|5 Clubs|Fairways of Life|"
    r"30 for 30|E60|SC Featured|ESPN Films|UFC Unleashed|Poker|BET Live|Daily Wager|"
    r"Championship Update|Postgame|Pregame|Preview|Highlights|The Drop|In the Crease|ESPN FC|"
    r"No-Contest Wrestling|PFL Fight Central|Pardon the Interruption"
    r")\b",
    flags=re.IGNORECASE,
)

TV_GUIDE_INCLUDE_EVENT_RE = re.compile(
    r"\b("
    r"NFL Football|UFL Football|College Football|NBA Basketball|WNBA Basketball|NHL Hockey|MLB Baseball|"
    r"College Baseball|NCAA Baseball|College Softball|Softball|College Basketball|"
    r"Women.?s College World Series|Womens College World Series|WCWS|College World Series|"
    r"Soccer|UEFA|Premier League|MLS|NWSL|Concacaf|CONCACAF|World Cup|International Soccer|"
    r"PGA Tour Golf|LPGA Tour Golf|PGA European Tour Golf|DP World Tour Golf|Korn Ferry|"
    r"PGA Tour Champions|Champions Tour Golf|College Golf|LIV Golf|Golf|"
    r"NCAA Men's|NCAA Women.?s|NCAA Womens|National Championship|U\.?S\.? Open|The Open|Masters|PGA Championship|"
    r"Tennis|French Open|Wimbledon|US Open Tennis|Australian Open|"
    r"NASCAR|Formula 1|IndyCar|MotoGP|Auto Racing|Motorsports|Lacrosse|Volleyball|"
    r"Rugby|MMA|UFC|PFL|Boxing|Horse Racing|Drag Racing|NHRA"
    r")\b",
    flags=re.IGNORECASE,
)

# Short tags for the grid
SPORT_TAGS = {
    "football/nfl": "NFL",
    "basketball/nba": "NBA",
    "hockey/nhl": "NHL",
    "baseball/mlb": "MLB",
    "baseball/college-softball": "Softball",
    "football/college-football": "FBS FB",
    "basketball/mens-college-basketball": "M CBB",
    "basketball/womens-college-basketball": "W CBB",
    "soccer/usa.1": "Soccer - MLS",
    "soccer/eng.1": "Soccer - EPL",
    "soccer/uefa.champions": "Soccer - UCL",
    "golf/pga": "PGA Tour",
    "golf/champions-tour": "Champions Tour",
    "golf/lpga": "LPGA",
    "golf/liv": "LIV Golf",
    "golf/eur": "European Tour",
    "racing/f1": "F1",
    "tennis/atp": "ATP",
    "tennis/wta": "WTA",
}

# ---- Favorites config
# Favorite pro teams now support optional row labels and row colors.
# You can still use the old tuple style:
#     ("Boston Bruins", ["hockey/nhl"])
# but the dict style below lets each favorite row carry its own colors.
FAVORITE_PRO_TEAMS = [
    # {
    #     "name": "Boston Bruins",
    #     "leagues": ["hockey/nhl"],
    #     "row_label": "Boston Bruins",
    #     "bg_color": "#FFB81C",
    #     "font_color": "#000000",
    # },
    # {
    #     "name": "Boston Celtics",
    #     "leagues": ["basketball/nba"],
    #     "row_label": "Boston Celtics",
    #     "bg_color": "#007A33",
    #     "font_color": "#FFFFFF",
    # },
    {
        "name": "Boston Red Sox",
        "leagues": ["baseball/mlb"],
        "row_label": "Boston\nRed Sox",
        "bg_color": "#BD3039",   # Red Sox red
        "font_color": "#FFFFFF",
    },
    {
        "name": "Detroit Tigers",
        "leagues": ["baseball/mlb"],
        "row_label": "Detroit\nTigers",
        "bg_color": "#0C2340",   # Tigers navy
        "font_color": "#FFFFFF",
    },
]

# Favorite school: name + leagues to scan (add/remove as needed)
FAVORITE_SCHOOL = {
    "name": "Michigan State",
    "row_label": "MSU\non B1G+",
    "leagues": [
        "football/college-football",
        "basketball/mens-college-basketball",
        "basketball/womens-college-basketball",
        "hockey/mens-college-hockey",
        "baseball/college-baseball",
        "baseball/college-softball",
        # "soccer/mens-college-soccer",
        # "soccer/womens-college-soccer",
        "volleyball/womens-college-volleyball",
        # add other college leagues here as you discover ESPN keys you care about
    ],
    # special row styling
    "bg_color": "#18453B",   # forest green
    "font_color": "#FFFFFF", # white
}

def _as_favorite_config(fav):
    """
    Normalize favorite-team config so older tuple entries still work.

    Supported:
      ("Boston Red Sox", ["baseball/mlb"])

    Preferred:
      {
          "name": "Boston Red Sox",
          "leagues": ["baseball/mlb"],
          "row_label": "Boston Red Sox",
          "bg_color": "#BD3039",
          "font_color": "#FFFFFF",
      }
    """
    if isinstance(fav, dict):
        return {
            "name": fav["name"],
            "leagues": fav["leagues"],
            "row_label": fav.get("row_label", fav["name"]),
            "bg_color": fav.get("bg_color", "#F5F5F5"),
            "font_color": fav.get("font_color", "#000000"),
        }

    team_name, league_keys = fav
    return {
        "name": team_name,
        "leagues": league_keys,
        "row_label": team_name,
        "bg_color": "#F5F5F5",
        "font_color": "#000000",
    }

# Map ESPN broadcast strings -> canonical channel labels (for the normal rows)
CHANNEL_ALIASES = {
    "abc": "ABC", "abc network": "ABC",
    "fox": "FOX", "fox network": "FOX",
    "cbs": "CBS", "cbs network": "CBS",
    "nbc": "NBC", "nbc network": "NBC", "nbc sports": "NBC",
    "nbcsn": "NBCSN", "nbc sports network": "NBCSN",

    "espn": "ESPN", "espn2": "ESPN2", "espnu": "ESPNU",
    "espn news": "ESPNews", "espnnews": "ESPNews", "espnews": "ESPNews",

    "fs1": "FS1", "fox sports 1": "FS1", "fox sports1": "FS1",

    "btn": "Big Ten", "big ten network": "Big Ten",
    "golf channel": "Golf", "golf chnl": "Golf", "golfchannel": "Golf",

    "usa": "USA", "usa network": "USA", "USA Network": "USA",
    "tnt": "TNT", "tnt hd": "TNT",
    "tbs": "TBS",
    "trutv": "truTV",
    "fanduel sn det": "FanDuel",
    "b1g+": "B1G+ APP",
    "peacock": "Peacock",
    "tigers tv": "Tigers TV", "tigerstv": "Tigers TV",
}

## TEST TO EXCLUDE STRERAMING SERVICES
EXCLUDE_STREAMING_KEYS = ("espn+", "espn plus", "espn app", "paramount+", "paramount plus",
                            # "peacock+", "peacock premium"
    )
LOCAL_TZ = ZoneInfo("America/Detroit")

# ========================== IMPORTS ==========================
import re
import math
import pandas as pd
import requests
try:
    from bs4 import BeautifulSoup
except ImportError:
    BeautifulSoup = None
from collections import defaultdict
from urllib.parse import urlencode

# ========================== HELPERS ==========================
def to_local_timestr(iso_str: str):
    if not iso_str:
        return None, None
    ts_utc = pd.to_datetime(iso_str, errors="coerce", utc=True)
    if pd.isna(ts_utc):
        return None, None
    ts_local = ts_utc.tz_convert(LOCAL_TZ).tz_localize(None)
    try:
        hm = ts_local.strftime("%-I:%M%p").lower()
    except ValueError:
        hm = ts_local.strftime("%#I:%M%p").lower()
    return ts_local, hm

def normalize_key(s: str) -> str:
    return re.sub(r"[^a-z0-9+ ]", "", s.lower()).strip()

def _split_broadcast_tokens(raw: str) -> list[str]:
    s = re.sub(r"[\/&]| and ", ",", raw, flags=re.IGNORECASE)
    parts = [p.strip() for p in s.split(",")]
    return [p for p in parts if p]

def normalize_channel_name(name: str) -> str | None:
    if not name:
        return None
    lk = name.lower()
    if any(x in lk for x in EXCLUDE_STREAMING_KEYS):
        return None
    return CHANNEL_ALIASES.get(normalize_key(name))

def sport_is_womens(league_key: str) -> bool:
    return "womens" in league_key or "college-softball" in league_key

def team_label(c: dict) -> str:
    t = (c or {}).get("team", {}) or {}
    rank = (c or {}).get("curatedRank", {}).get("current")
    nm = t.get("displayName") or t.get("shortDisplayName") or t.get("name") or ""
    return (f"#{rank} " if rank and rank != 99 else "") + nm

def build_title(comp: dict, ev: dict | None = None, prefix_womens=False) -> str:
    """Build a compact title for team-vs-team events, with a fallback for golf/event-style feeds."""
    comps = comp.get("competitors", []) or []
    by_side = {c.get("homeAway"): c for c in comps if c.get("homeAway")}
    away = team_label(by_side.get("away", {}))
    home = team_label(by_side.get("home", {}))

    if away and home:
        title = f"{away} at {home}".strip()
    else:
        # Golf and some ESPN event feeds do not have home/away teams.
        title = (comp.get("shortName") or comp.get("name")
                 or (ev or {}).get("shortName") or (ev or {}).get("name")
                 or "Untitled event")
        status_detail = ((comp.get("status") or {}).get("type") or {}).get("shortDetail")
        if status_detail and status_detail.lower() not in title.lower():
            title = f"{title} — {status_detail}"

    # If an event-style feed carries sponsor filler, keep the printable title compact.
    if "presented by" in title.lower() or "sponsored by" in title.lower() or "powered by" in title.lower():
        title = _strip_event_sponsor_filler(title)

    if prefix_womens:
        title = f"(W) {title}"
    return title

def _scoreboard_url(league_key: str, day: date) -> str:
    """Return the ESPN scoreboard URL for a league/date, with per-league overrides."""
    ymd = day.strftime("%Y%m%d")
    base_url = SCOREBOARD_URL_OVERRIDES.get(
        league_key,
        f"https://site.api.espn.com/apis/site/v2/sports/{league_key}/scoreboard"
    )
    sep = "&" if "?" in base_url else "?"
    return f"{base_url}{sep}{urlencode({'dates': ymd})}"

def fetch_day_sport_multi(day: date, league_keys: list[str]) -> list[dict]:
    last_err = None
    for k in league_keys:
        url = _scoreboard_url(k, day)
        try:
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            return r.json().get("events", []) or []
        except Exception as e:
            last_err = e
            continue
    if last_err:
        print(f"[WARN] {day} {'/'.join(league_keys)} fetch failed: {last_err}")
    return []


def event_occurs_on_day(ev: dict, comp: dict, day: date) -> bool:
    """True when a team game starts that day, or a golf/tournament event spans that day."""
    start_raw = comp.get("date") or comp.get("startDate") or ev.get("date")
    end_raw = comp.get("endDate") or ev.get("endDate")
    start_dt, _ = to_local_timestr(start_raw)
    if start_dt is None:
        return False
    if start_dt.date() == day:
        return True
    if end_raw:
        end_dt, _ = to_local_timestr(end_raw)
        if end_dt is not None:
            return start_dt.date() <= day <= end_dt.date()
    return False

def extract_broadcast_tokens(ev: dict, comp: dict) -> list[str]:
    candidates = []
    for b in (comp.get("broadcasts") or []):
        media = b.get("media") or {}
        for k in ("shortName", "name"):
            if media.get(k): candidates.append(str(media[k]))
        for k in ("shortName", "name"):
            if b.get(k): candidates.append(str(b[k]))
        for n in (b.get("names") or []):
            candidates.append(str(n))
    for gb in (ev.get("geoBroadcasts") or []):
        chan = gb.get("media", {}).get("shortName") or gb.get("media", {}).get("name")
        if chan: candidates.append(str(chan))
    b = comp.get("broadcast") or {}
    if isinstance(b, dict):
        for k in ("shortName", "name"):
            if b.get(k): candidates.append(str(b[k]))
    tokens, seen = [], set()
    for raw in candidates:
        for tok in _split_broadcast_tokens(raw):
            nk = normalize_key(tok)
            if nk and nk not in seen:
                seen.add(nk)
                tokens.append(tok)
    return tokens

def get_event_tag(league_key: str, sport_label: str) -> str:
    tag = SPORT_TAGS.get(league_key)
    if not tag:
        fallback = {
            "NBA":"NBA","NHL":"NHL","NFL":"NFL","MLB":"MLB","CFB":"CFB",
            "MBB":"MBB","WBB":"WBB","MLS":"MLS","EPL":"EPL","UCL":"UCL",
            "PGA":"PGA","LPGA":"LPGA","LIV":"LIV","DPW":"DPW"
        }.get(sport_label, "SPORT")
        tag = fallback
    return f"({tag})"

def _break_after_at(title: str) -> str:
    return re.sub(r"\s+(at|vs\.?|v\.)\s+", r" \1\n", title, count=1, flags=re.IGNORECASE)


def _format_title_for_grid_cell(title: str, tag: str = "") -> str:
    """Apply final printable line-break rules for grid-cell titles.

    Most team-vs-team rows get a subtle break before "at/vs.".
    Golf and racing titles often arrive as "Event — Round" or
    "Series — Race"; for the printed grid, replace that long dash
    with a real line break instead of leaving the dash as filler.
    """
    title = str(title or "").strip()
    tag_key = (tag or "").strip().strip("()").upper()

    dash_break_tags = {
        "PGA", "PGA TOUR", "LPGA", "LIV", "DPW", "KFT",
        "GOLF", "COLLEGE GOLF", "CHAMPIONS TOUR", "RACING",
    }

    if tag_key in dash_break_tags:
        title = re.sub(r"\s+[—–]\s+", "\n", title)

    if "\n" in title:
        return title
    return _break_after_at(title)

def _comp_has_team(comp: dict, needle: str) -> bool:
    """case-insensitive contains on team display names."""
    if not needle:
        return False
    needle = needle.lower()
    for c in (comp.get("competitors") or []):
        t = (c or {}).get("team", {}) or {}
        for f in ("displayName", "shortDisplayName", "name"):
            val = (t.get(f) or "").lower()
            if val and needle in val:
                return True
    return False


# ========================== TV GUIDE BACKUP HELPERS ==========================
AYWT_DATE_HEADER_RE = re.compile(
    r"^(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),\s+([A-Za-z]{3,9})\.?\s+(\d{1,2})$",
    flags=re.IGNORECASE,
)
AYWT_LISTING_RE = re.compile(r"^\*?\s*(\d{1,2}:\d{2}\s*[ap])\s*-\s*(.+)$", flags=re.IGNORECASE)
AYWT_MONTH_LOOKUP = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}

TV_GUIDE_PARSED_ITEMS = []
TV_GUIDE_REJECTED_ITEMS = []
TV_GUIDE_KEPT_ITEMS = []

TV_GUIDE_DIAG_COLUMNS = [
    "source", "channel", "date", "start_dt", "time_str", "raw_time", "category",
    "title", "description", "url", "decision_reason", "inferred_tag"
]


def _clean_guide_line(line: str) -> str:
    line = re.sub(r"\s+", " ", str(line or "")).strip()
    line = re.sub(r"^#{1,6}\s*", "", line).strip()
    if not line or line.lower().startswith("image:"):
        return ""
    return line


def _extract_text_lines_from_html(html: str) -> list[str]:
    if BeautifulSoup is not None:
        soup = BeautifulSoup(html, "html.parser")
        for tag in soup(["script", "style", "noscript", "svg"]):
            tag.decompose()

        # Are You Watching This?! schedule rows are usually <li> items. Pulling
        # those with a space separator preserves compact rows like
        # "7:30p - MLB Baseball - Team A vs. Team B" even when links/spans split
        # the text internally. We also pull date headers before the broad text pass.
        explicit_lines = []
        for tag in soup.find_all(["h2", "h3", "h4", "li"]):
            txt = _clean_guide_line(tag.get_text(" ", strip=True))
            if txt and (AYWT_DATE_HEADER_RE.match(txt) or AYWT_LISTING_RE.match(txt)):
                explicit_lines.append(txt)

        for tag in soup.find_all(["br", "p", "div", "li", "h1", "h2", "h3", "h4", "h5", "h6"]):
            tag.insert_before("\n")
            tag.insert_after("\n")
        raw_lines = explicit_lines + soup.get_text("\n").splitlines()
    else:
        text = re.sub(r"(?i)<br\s*/?>", "\n", html)
        text = re.sub(r"(?i)</(p|div|li|h[1-6])>", "\n", text)
        text = re.sub(r"<[^>]+>", " ", text)
        raw_lines = text.splitlines()

    lines = []
    for line in raw_lines:
        cleaned = _clean_guide_line(line)
        if cleaned:
            lines.append(cleaned)
    return lines


def _resolve_aywt_date(month_text: str, day_num: int) -> date | None:
    month = AYWT_MONTH_LOOKUP.get(str(month_text).lower().strip("."))
    if not month:
        return None

    # Station pages cover a moving two-week window. Resolve month/day to the
    # year nearest the requested START_DATE window, handling Dec/Jan wraparound.
    candidates = []
    for year in (START_DATE.year - 1, START_DATE.year, START_DATE.year + 1):
        try:
            d = date(year, month, int(day_num))
            candidates.append(d)
        except ValueError:
            pass
    if not candidates:
        return None

    # Prefer a date that falls near the requested window.
    window_start = START_DATE - timedelta(days=10)
    window_end = START_DATE + timedelta(days=30)
    in_window = [d for d in candidates if window_start <= d <= window_end]
    if in_window:
        return min(in_window, key=lambda d: abs((d - START_DATE).days))
    return min(candidates, key=lambda d: abs((d - START_DATE).days))


def _parse_aywt_date_header(line: str) -> date | None:
    m = AYWT_DATE_HEADER_RE.match(line or "")
    if not m:
        return None
    return _resolve_aywt_date(m.group(2), int(m.group(3)))


def _parse_aywt_time(day: date, time_text: str) -> tuple[datetime | None, str | None]:
    m = re.match(r"^(\d{1,2}):(\d{2})\s*([ap])$", str(time_text or "").strip(), flags=re.I)
    if not m:
        return None, None
    hour = int(m.group(1))
    minute = int(m.group(2))
    period = m.group(3).lower()
    if period == "a" and hour == 12:
        hour = 0
    elif period == "p" and hour != 12:
        hour += 12
    dt = datetime(day.year, day.month, day.day, hour, minute)
    try:
        hm = dt.strftime("%-I:%M%p").lower()
    except ValueError:
        hm = dt.strftime("%#I:%M%p").lower()
    return dt, hm


def _split_aywt_listing(rest: str) -> tuple[str, str, str]:
    """Return (category, title, display_title) from AYWT's 'Sport - Program' text."""
    rest = re.sub(r"\s+", " ", rest or "").strip()
    if " - " in rest:
        category, title = rest.split(" - ", 1)
        category = category.strip()
        title = title.strip()
        display = f"{category} — {title}" if title else category
        return category, title or category, display
    return "", rest, rest


def _strip_tvguide_venue_suffix(text: str) -> str:
    """Remove venue clutter from guide titles without damaging normal team names.

    AYWT often emits titles like:
        NHL Hockey — Vegas Golden Knights vs. Colorado Avalanche at Ball Arena
    For a compact printed schedule, the venue is usually noise. We only remove
    the final venue phrase when the title has a true "vs." matchup first, so
    entries like "NASCAR ... Race at Michigan" are left alone.
    """
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if not TV_GUIDE_STRIP_VENUE_FROM_TITLES or not text:
        return text
    if not re.search(r"\bvs\.?\b", text, flags=re.I):
        return text
    # Remove final " at Venue" after a vs. matchup. This intentionally does not
    # touch scoreboard-style titles such as "Detroit Tigers at Chicago White Sox"
    # because this function is only called for TV-guide rows.
    return re.sub(r"(?i)(\bvs\.?\s+.+?)\s+at\s+[^—|]+$", r"\1", text).strip()


def _strip_event_sponsor_filler(text: str) -> str:
    """Remove sponsor filler such as 'presented by Workday' from event titles."""
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if not text:
        return text

    # Handles examples like these while keeping any later round text:
    #   the Memorial Tournament presented by Workday — Third Round
    #   BMW Charity Pro-Am presented by TD SYNNEX
    text = re.sub(
        r"(?i)\s+\b(?:presented|sponsored|powered)\s+by\s+[^—|:,;()]+(?=\s*(?:[—|:,;()]|$))",
        "",
        text,
    )
    # Normalize spacing around separators in case the sponsor phrase consumed
    # the space before a dash/colon/comma.
    text = re.sub(r"\s*([—|:,;])\s*", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip(" -–—")
    return text


def _simplify_tvguide_title(text: str) -> str:
    """Compact TV-guide titles where the tag already carries the sport/league context."""
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if not text:
        return text

    m = re.match(
        r"(?i)^(Men(?:'s)?|Women(?:'s)?|Womens)\s+International\s+Soccer\s+Friendlies?\s*[—-]\s*(.+)$",
        text,
    )
    if m:
        gender_raw = m.group(1).lower()
        gender = "(M)" if gender_raw.startswith("men") else "(W)"
        matchup = m.group(2).strip()
        return f"{gender} International Friendly" + "\n" + matchup

    m = re.match(r"(?i)^WNBA\s+Basketball\s*[—-]\s*(.+)$", text)
    if m:
        return m.group(1).strip()

    return text


def _compact_multiline_for_grid(text: str, max_chars_per_line: int = 95) -> str:
    """Shorten explicit multiline titles without collapsing the line breaks."""
    lines = []
    for line in str(text or "").splitlines():
        clean = re.sub(r"\s+", " ", line).strip()
        if clean:
            lines.append(_shorten_for_grid(clean, max_chars=max_chars_per_line))
    return "\n".join(lines)


def _is_probable_tvguide_replay(item: dict, tag: str) -> tuple[bool, str]:
    """Heuristic replay filter for station-guide rows.

    AYWT does not consistently mark replays in the text we scrape. Overnight
    listings before the morning programming window are usually replays for the
    broadcast-grid use case, especially for NHL/NBA/MLB/FS1 team events. We keep
    this deliberately simple and configurable.
    """
    if not TV_GUIDE_DROP_EARLY_MORNING_REPLAYS:
        return False, ""
    start_dt = item.get("start_dt")
    if not isinstance(start_dt, datetime):
        return False, ""
    if start_dt.hour < int(TV_GUIDE_EARLY_MORNING_CUTOFF_HOUR):
        return True, f"probable_replay_before_{TV_GUIDE_EARLY_MORNING_CUTOFF_HOUR}am"
    return False, ""


def parse_aywt_schedule(channel: str, url: str) -> list[dict]:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
    }
    try:
        r = requests.get(url, headers=headers, timeout=25)
        r.raise_for_status()
    except Exception as e:
        print(f"[WARN] TV guide fetch failed for {channel} ({url}): {e}")
        return []

    lines = _extract_text_lines_from_html(r.text)
    out = []
    seen_rows = set()
    current_date = None

    for line in lines:
        parsed_date = _parse_aywt_date_header(line)
        if parsed_date is not None:
            current_date = parsed_date
            continue

        if current_date not in DAY_DATES:
            continue

        m = AYWT_LISTING_RE.match(line)
        if not m:
            continue

        raw_time = m.group(1).lower().replace(" ", "")
        rest = m.group(2).strip()
        start_dt, hm = _parse_aywt_time(current_date, raw_time)
        if start_dt is None:
            continue

        category, title, display_title = _split_aywt_listing(rest)
        row_key = (channel, current_date, raw_time, display_title)
        if row_key in seen_rows:
            continue
        seen_rows.add(row_key)
        out.append({
            "source": "tvguide_aywt",
            "channel": channel,
            "date": current_date,
            "start_dt": start_dt,
            "time_str": hm,
            "raw_time": raw_time,
            "category": category,
            "title": display_title,
            "description": title if title != display_title else "",
            "url": url,
        })

    return out


def infer_tvguide_tag(title: str, desc: str = "", category: str = "") -> str:
    text = f"{category} {title} {desc}".lower()
    text_ascii = (
        text.replace("’", "'")
            .replace("women’s", "women's")
            .replace("womens", "women's")
            .replace("u.s.", "us")
    )
    if "women's college world series" in text_ascii or "wcws" in text_ascii:
        return "(Softball)"
    if "softball" in text_ascii:
        return "(Softball)"
    if "college world series" in text_ascii or "college baseball" in text_ascii or "ncaa baseball" in text_ascii:
        return "(NCAA Baseball)"
    if "mlb" in text_ascii or "baseball" in text_ascii:
        return "(MLB)"
    # Check WNBA before NBA because "wnba basketball" contains the substring
    # "nba basketball".
    if "wnba" in text_ascii:
        return "(WNBA)"
    if "nba finals" in text_ascii or "nba basketball" in text_ascii:
        return "(NBA)"
    if "stanley cup" in text_ascii or "nhl" in text_ascii or "hockey" in text_ascii:
        return "(NHL)"
    if "college football" in text_ascii:
        return "(FBS FB)"
    if "ufl football" in text_ascii:
        return "(UFL)"
    if "nfl" in text_ascii or "football" in text_ascii:
        return "(NFL)"
    if "college golf" in text_ascii:
        return "(College Golf)"
    if "lpga" in text_ascii or "women's open" in text_ascii:
        return "(LPGA)"
    if "champions tour" in text_ascii or ("champions" in text_ascii and "golf" in text_ascii):
        return "(Champions Tour)"
    if "pga european tour" in text_ascii or "dp world" in text_ascii:
        return "(DPW)"
    if "korn ferry" in text_ascii:
        return "(KFT)"
    if "liv golf" in text_ascii:
        return "(LIV)"
    if "pga tour golf" in text_ascii or "memorial tournament" in text_ascii or "masters" in text_ascii or "pga championship" in text_ascii or "us open" in text_ascii or "the open" in text_ascii:
        return "(PGA Tour)"
    if "golf" in text_ascii:
        return "(Golf)"
    if "soccer" in text_ascii or "uefa" in text_ascii or "premier league" in text_ascii or "mls" in text_ascii or "concacaf" in text_ascii or "world cup" in text_ascii:
        return "(Soccer)"
    if "tennis" in text_ascii or "french open" in text_ascii or "wimbledon" in text_ascii:
        return "(Tennis)"
    if "nascar" in text_ascii or "formula 1" in text_ascii or "indycar" in text_ascii or "motogp" in text_ascii or "auto racing" in text_ascii or "nhra" in text_ascii or "drag racing" in text_ascii:
        return "(Racing)"
    if "volleyball" in text_ascii:
        return "(Volleyball)"
    if "rugby" in text_ascii:
        return "(Rugby)"
    if "mma" in text_ascii or "ufc" in text_ascii or "pfl" in text_ascii or "boxing" in text_ascii:
        return "(Fighting)"
    if "horse racing" in text_ascii:
        return "(Horse Racing)"
    return "(TV)"


def is_live_tvguide_event(item: dict) -> tuple[bool, str]:
    title = item.get("title") or ""
    desc = item.get("description") or ""
    category = item.get("category") or ""
    text = f"{category} {title} {desc}"

    if TV_GUIDE_EXCLUDE_TITLE_RE.search(text):
        return False, "excluded_title"

    # Strict mode is for actual games/events, not studio shows or shoulder programming.
    if TV_GUIDE_EVENT_STRICTNESS == "live_only":
        if re.search(r"\b(Pregame|Postgame|Preview|Highlights|Live From|Tip-Off|Championship Update)\b", text, flags=re.I):
            return False, "strict_studio_or_shoulder"

    tag = infer_tvguide_tag(title, desc, category)
    replay_drop, replay_reason = _is_probable_tvguide_replay(item, tag)
    if replay_drop:
        return False, replay_reason

    if tag != "(TV)":
        return True, "tag_inferred"

    if TV_GUIDE_INCLUDE_EVENT_RE.search(text):
        return True, "include_regex"

    return False, "no_live_event_match"


def _shorten_for_grid(text: str, max_chars: int = 105) -> str:
    text = re.sub(r"\s+", " ", text or "").strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0].rstrip(".,;:") + "…"


def tvguide_display_title(item: dict) -> str:
    # AYWT already provides compact sport/category + program text, but its event
    # descriptions often include venue clutter that makes printed cells bulky.
    title = item.get("title") or "Untitled TV event"
    title = _strip_tvguide_venue_suffix(title)
    title = _strip_event_sponsor_filler(title)
    title = _simplify_tvguide_title(title)
    if "\n" in title:
        return _compact_multiline_for_grid(title, max_chars_per_line=95)
    return _shorten_for_grid(title, max_chars=130)


def _write_tvguide_debug_csvs():
    if not TV_GUIDE_DEBUG_OUTPUTS:
        return
    try:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        parsed_df = pd.DataFrame(TV_GUIDE_PARSED_ITEMS)
        kept_df = pd.DataFrame(TV_GUIDE_KEPT_ITEMS)
        rejected_df = pd.DataFrame(TV_GUIDE_REJECTED_ITEMS)
        for df_out in (parsed_df, kept_df, rejected_df):
            for col in TV_GUIDE_DIAG_COLUMNS:
                if col not in df_out.columns:
                    df_out[col] = pd.Series(dtype="object")
        parsed_df[TV_GUIDE_DIAG_COLUMNS].to_csv(os.path.join(OUTPUT_DIR, "tvguide_parsed_all.csv"), index=False)
        kept_df[TV_GUIDE_DIAG_COLUMNS].to_csv(os.path.join(OUTPUT_DIR, "tvguide_kept.csv"), index=False)
        rejected_df[TV_GUIDE_DIAG_COLUMNS].to_csv(os.path.join(OUTPUT_DIR, "tvguide_rejected.csv"), index=False)
    except Exception as e:
        print(f"[WARN] Could not write TV guide debug CSVs: {e}")


def build_tvguide_rows() -> list[dict]:
    global TV_GUIDE_PARSED_ITEMS, TV_GUIDE_REJECTED_ITEMS, TV_GUIDE_KEPT_ITEMS
    if not USE_TV_GUIDE_BACKUP:
        return []
    active_channels = CHANNEL_MAPS[ACTIVE_CHANNEL_MAP_NAME]
    guide_rows = []
    seen_guide_title_day = set()
    TV_GUIDE_PARSED_ITEMS = []
    TV_GUIDE_REJECTED_ITEMS = []
    TV_GUIDE_KEPT_ITEMS = []

    for channel, url in TV_GUIDE_CHANNEL_URLS.items():
        if channel not in active_channels:
            continue
        parsed_items = parse_aywt_schedule(channel, url)
        print(f"TV guide {channel}: parsed {len(parsed_items)} candidate station listings")
        for item in parsed_items:
            TV_GUIDE_PARSED_ITEMS.append(item.copy())
            keep, reason = is_live_tvguide_event(item)
            diagnostic = item.copy()
            diagnostic["decision_reason"] = reason
            diagnostic["inferred_tag"] = infer_tvguide_tag(
                item.get("title", ""),
                item.get("description", ""),
                item.get("category", ""),
            )
            if not keep:
                TV_GUIDE_REJECTED_ITEMS.append(diagnostic)
                continue
            tag = diagnostic["inferred_tag"]
            display_title = tvguide_display_title(item)
            if TV_GUIDE_DEDUPLICATE_SAME_TITLE_BY_DAY:
                duplicate_key = (
                    channel,
                    item["date"],
                    tag,
                    normalize_key(display_title),
                )
                if duplicate_key in seen_guide_title_day:
                    diagnostic["decision_reason"] = "duplicate_same_title_channel_day"
                    TV_GUIDE_REJECTED_ITEMS.append(diagnostic)
                    continue
                seen_guide_title_day.add(duplicate_key)

            row = {
                "col_label": _col_label(item["date"]),
                "date": item["date"],
                "time_str": item["time_str"],
                "start_dt": item["start_dt"],
                "sport": "TV Guide",
                "league_key": "tvguide_aywt",
                "tag": tag,
                "channel": channel,
                "channel_num": active_channels[channel],
                "title": display_title,
                "fav_row": None,
                "source": "tvguide",
            }
            guide_rows.append(row)
            kept_diag = diagnostic.copy()
            kept_diag.update(row)
            TV_GUIDE_KEPT_ITEMS.append(kept_diag)

    print(
        f"TV guide backup parsed {len(TV_GUIDE_PARSED_ITEMS)} total listings; "
        f"kept {len(TV_GUIDE_KEPT_ITEMS)} candidate event rows; "
        f"rejected {len(TV_GUIDE_REJECTED_ITEMS)}"
    )

    _write_tvguide_debug_csvs()

    if guide_rows:
        guide_summary = pd.DataFrame(guide_rows).groupby(["channel", "tag"]).size().reset_index(name="rows")
        print("TV guide kept rows by channel/tag:")
        print(guide_summary.to_string(index=False))
    else:
        print("TV guide produced no kept rows. Check tvguide_parsed_all.csv and tvguide_rejected.csv in output/.")

    return guide_rows



# ========================== SPORTS MEDIA WATCH GOLF HELPERS ==========================
SMW_DATE_HEADER_RE = re.compile(
    r"^(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),\s+([A-Za-z]{3,9})\.?\s+(\d{1,2})(?:,\s*(\d{4}))?$",
    flags=re.IGNORECASE,
)
SMW_TIME_RE = re.compile(
    r"^(\d{1,2}:\d{2}\s*(?:a|p|am|pm))(?:\s*[-–—]\s*(\d{1,2}:\d{2}\s*(?:a|p|am|pm)))?$",
    flags=re.IGNORECASE,
)
SMW_CHANNEL_HINT_RE = re.compile(
    r"\b(ABC|CBS|NBC|FOX|USA\s+Network|USA|Golf\s+Channel|GOLF|ESPN\+?|ESPN2|ESPNU|ESPNews|FS1|FS2|TNT|TBS|truTV|Peacock|NBCSN|NBC\s+Sports\s+Network|Paramount\+|CBS\s+Sports\s+Network)\b",
    flags=re.IGNORECASE,
)
SMW_NOISE_LINES = {
    "time game / tv", "time round / tv", "filterdatetext searchall", "apply",
    "no more upcoming games to show.", "show past games", "no more games to show.",
}
SMW_PARSED_ITEMS = []
SMW_KEPT_ITEMS = []
SMW_REJECTED_ITEMS = []
SMW_DIAG_COLUMNS = [
    "source", "page_label", "channel", "date", "start_dt", "time_str", "raw_time",
    "title", "round", "location", "network_line", "url", "decision_reason", "inferred_tag"
]


def _parse_smw_date_header(line: str) -> date | None:
    line = _clean_guide_line(line)
    if not line:
        return None
    if line.lower() == "today":
        return datetime.now(LOCAL_TZ).date()
    if line.lower() == "tomorrow":
        return datetime.now(LOCAL_TZ).date() + timedelta(days=1)
    m = SMW_DATE_HEADER_RE.match(line)
    if not m:
        return None
    month = AYWT_MONTH_LOOKUP.get(m.group(2).lower().strip("."))
    if not month:
        return None
    day_num = int(m.group(3))
    year = int(m.group(4)) if m.group(4) else START_DATE.year
    try:
        return date(year, month, day_num)
    except ValueError:
        return None


def _parse_smw_time(day: date, time_text: str) -> tuple[datetime | None, str | None]:
    m = SMW_TIME_RE.match(str(time_text or "").strip())
    if not m:
        return None, None
    start_text = m.group(1).lower().replace(" ", "")
    start_text = start_text.replace("am", "a").replace("pm", "p")
    dt_naive, hm = _parse_aywt_time(day, start_text)
    if dt_naive is None:
        return None, None
    # Sports Media Watch states its schedules in Eastern time. Convert to the
    # worksheet's local timezone; this is currently also Eastern for StoryPoint EL,
    # but keeping the conversion makes the helper portable.
    eastern = ZoneInfo("America/New_York")
    dt_local = dt_naive.replace(tzinfo=eastern).astimezone(LOCAL_TZ).replace(tzinfo=None)
    try:
        hm = dt_local.strftime("%-I:%M%p").lower()
    except ValueError:
        hm = dt_local.strftime("%#I:%M%p").lower()
    return dt_local, hm


def _looks_like_smw_channel_line(line: str) -> bool:
    if not line:
        return False
    return bool(SMW_CHANNEL_HINT_RE.search(line))


def _clean_smw_channel_line(line: str) -> str:
    line = re.sub(r"\s+", " ", str(line or "")).strip()
    if GOLF_SCHEDULE_US_ONLY:
        line = line.split("|", 1)[0].strip()
    return line


def _channel_tokens_from_smw(line: str) -> list[str]:
    line = _clean_smw_channel_line(line)
    # Normalize common combined text before the generic splitter.
    line = line.replace("USA Network", "USA")
    line = line.replace("Golf Channel", "Golf")
    line = line.replace("NBC Sports Network", "NBCSN")
    tokens = []
    for raw in _split_broadcast_tokens(line):
        raw = raw.strip()
        # Remove parenthetical notes and affiliate/streaming clutter after spaces.
        raw = re.sub(r"\s*\([^)]*\)", "", raw).strip()
        if not raw:
            continue
        key = normalize_key(raw)
        if any(excl in key for excl in GOLF_SCHEDULE_EXCLUDE_CHANNEL_KEYS):
            continue
        tokens.append(raw)
    return tokens


def _clean_smw_title(title: str, round_text: str = "") -> str:
    title = re.sub(r"\s+", " ", str(title or "")).strip()
    round_text = re.sub(r"\s+", " ", str(round_text or "")).strip()
    title = _strip_event_sponsor_filler(title)
    round_text = _strip_event_sponsor_filler(round_text)
    if round_text and normalize_key(round_text) not in normalize_key(title):
        title = f"{title} — {round_text}"
    title = _strip_event_sponsor_filler(title)
    # Keep tournament/round but intentionally omit location; it adds bulk and
    # is rarely needed in a printable TV grid.
    return _shorten_for_grid(title, max_chars=130)


def parse_smw_golf_schedule(page_label: str, url: str) -> list[dict]:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
    }
    try:
        r = requests.get(url, headers=headers, timeout=25)
        r.raise_for_status()
    except Exception as e:
        print(f"[WARN] SMW golf schedule fetch failed for {page_label} ({url}): {e}")
        return []

    lines = _extract_text_lines_from_html(r.text)
    out = []
    seen = set()
    current_date = None
    i = 0
    while i < len(lines):
        line = _clean_guide_line(lines[i])
        parsed_date = _parse_smw_date_header(line)
        if parsed_date is not None:
            current_date = parsed_date
            i += 1
            continue

        if current_date not in DAY_DATES:
            i += 1
            continue

        tm = SMW_TIME_RE.match(line)
        if not tm:
            i += 1
            continue

        start_dt, hm = _parse_smw_time(current_date, tm.group(1))
        if start_dt is None:
            i += 1
            continue

        raw_time = line
        title = ""
        round_text = ""
        location = ""
        network_line = ""

        j = i + 1
        details = []
        # Grab the row payload until the first network/channel line or the next
        # time/date marker. SMW pages usually expose: title, round, location, networks.
        while j < len(lines):
            nxt = _clean_guide_line(lines[j])
            if not nxt or normalize_key(nxt) in SMW_NOISE_LINES:
                j += 1
                continue
            if _parse_smw_date_header(nxt) is not None or SMW_TIME_RE.match(nxt):
                break
            if _looks_like_smw_channel_line(nxt):
                network_line = _clean_smw_channel_line(nxt)
                j += 1
                break
            details.append(nxt)
            j += 1

        if details:
            title = details[0]
        if len(details) >= 2:
            round_text = details[1]
        if len(details) >= 3:
            location = details[2]

        if not title or not network_line:
            i = max(j, i + 1)
            continue

        display_title = _clean_smw_title(title, round_text)
        for raw_channel in _channel_tokens_from_smw(network_line):
            canon = normalize_channel_name(raw_channel)
            if not canon:
                continue
            row_key = (page_label, current_date, raw_time, canon, display_title)
            if row_key in seen:
                continue
            seen.add(row_key)
            out.append({
                "source": "golf_smw",
                "page_label": page_label,
                "channel": canon,
                "date": current_date,
                "start_dt": start_dt,
                "time_str": hm,
                "raw_time": raw_time,
                "title": display_title,
                "round": round_text,
                "location": location,
                "network_line": network_line,
                "url": url,
                "inferred_tag": infer_tvguide_tag(display_title, round_text, "Golf"),
            })
        i = max(j, i + 1)

    return out


def _write_smw_debug_csvs():
    if not GOLF_SCHEDULE_DEBUG_OUTPUTS:
        return
    try:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        parsed_df = pd.DataFrame(SMW_PARSED_ITEMS)
        kept_df = pd.DataFrame(SMW_KEPT_ITEMS)
        rejected_df = pd.DataFrame(SMW_REJECTED_ITEMS)
        for df_out in (parsed_df, kept_df, rejected_df):
            for col in SMW_DIAG_COLUMNS:
                if col not in df_out.columns:
                    df_out[col] = pd.Series(dtype="object")
        parsed_df[SMW_DIAG_COLUMNS].to_csv(os.path.join(OUTPUT_DIR, "golf_smw_parsed_all.csv"), index=False)
        kept_df[SMW_DIAG_COLUMNS].to_csv(os.path.join(OUTPUT_DIR, "golf_smw_kept.csv"), index=False)
        rejected_df[SMW_DIAG_COLUMNS].to_csv(os.path.join(OUTPUT_DIR, "golf_smw_rejected.csv"), index=False)
    except Exception as e:
        print(f"[WARN] Could not write SMW golf debug CSVs: {e}")


def build_smw_golf_rows() -> list[dict]:
    global SMW_PARSED_ITEMS, SMW_KEPT_ITEMS, SMW_REJECTED_ITEMS
    if not USE_GOLF_SCHEDULE_BACKUP:
        return []
    active_channels = CHANNEL_MAPS[ACTIVE_CHANNEL_MAP_NAME]
    SMW_PARSED_ITEMS = []
    SMW_KEPT_ITEMS = []
    SMW_REJECTED_ITEMS = []
    rows_out = []
    seen = set()

    for page_label, url in GOLF_SCHEDULE_URLS.items():
        parsed_items = parse_smw_golf_schedule(page_label, url)
        print(f"SMW golf {page_label}: parsed {len(parsed_items)} schedule-channel rows")
        for item in parsed_items:
            diagnostic = item.copy()
            SMW_PARSED_ITEMS.append(diagnostic.copy())
            channel = item.get("channel")
            if channel in GOLF_SCHEDULE_SKIP_CHANNELS:
                diagnostic["decision_reason"] = "skip_channel_prefer_aywt"
                SMW_REJECTED_ITEMS.append(diagnostic)
                continue
            if channel not in active_channels:
                diagnostic["decision_reason"] = "channel_not_in_active_map"
                SMW_REJECTED_ITEMS.append(diagnostic)
                continue
            tag = item.get("inferred_tag") or infer_tvguide_tag(item.get("title", ""), item.get("round", ""), "Golf")
            row_key = (item["date"], channel, item["start_dt"], tag, normalize_key(item["title"]))
            if row_key in seen:
                diagnostic["decision_reason"] = "duplicate_smw_row"
                SMW_REJECTED_ITEMS.append(diagnostic)
                continue
            seen.add(row_key)
            row = {
                "col_label": _col_label(item["date"]),
                "date": item["date"],
                "time_str": item["time_str"],
                "start_dt": item["start_dt"],
                "sport": "Golf Schedule",
                "league_key": "golf_smw",
                "tag": tag,
                "channel": channel,
                "channel_num": active_channels[channel],
                "title": item["title"],
                "fav_row": None,
                "source": "golf_smw",
            }
            rows_out.append(row)
            kept = diagnostic.copy()
            kept["decision_reason"] = "kept"
            kept.update(row)
            SMW_KEPT_ITEMS.append(kept)

    print(
        f"SMW golf backup parsed {len(SMW_PARSED_ITEMS)} schedule-channel rows; "
        f"kept {len(SMW_KEPT_ITEMS)}; rejected {len(SMW_REJECTED_ITEMS)}"
    )
    _write_smw_debug_csvs()
    if rows_out:
        summary = pd.DataFrame(rows_out).groupby(["channel", "tag"]).size().reset_index(name="rows")
        print("SMW golf kept rows by channel/tag:")
        print(summary.to_string(index=False))
    return rows_out


def should_skip_scoreboard_for_normal_grid(league_keys: list[str]) -> bool:
    if USE_TV_GUIDE_BACKUP and not USE_GOLF_SCOREBOARD_FEEDS:
        return any(k in GOLF_SCOREBOARD_LEAGUES for k in league_keys)
    return False

# ========================== GATHER EVENTS ==========================
active_map = CHANNEL_MAPS[ACTIVE_CHANNEL_MAP_NAME]
unknown_broadcasts = defaultdict(int)
rows = []

# Normal channel-based events (filter to active_map)
for day in DAY_DATES:
    for sport_label, league_keys in SPORTS:
        if should_skip_scoreboard_for_normal_grid(league_keys):
            continue
        events = fetch_day_sport_multi(day, league_keys)
        if not events:
            continue
        for ev in events:
            for comp in (ev.get("competitions") or []):
                local_dt, hm = to_local_timestr(comp.get("date") or ev.get("date"))
                if local_dt is None or not event_occurs_on_day(ev, comp, day):
                    continue
                event_day = local_dt.date() if local_dt.date() == day else day
                col_label = _col_label(event_day)
                if local_dt.date() != day:
                    hm = "TBD"
                    sort_dt = datetime.combine(event_day, datetime.min.time())
                else:
                    sort_dt = local_dt

                # broadcasts -> channels
                channel_hits = set()
                for raw in extract_broadcast_tokens(ev, comp):
                    canon = normalize_channel_name(raw)
                    if canon:
                        channel_hits.add(canon)
                    else:
                        key = normalize_key(raw)
                        if key and not any(x in key for x in EXCLUDE_STREAMING_KEYS):
                            unknown_broadcasts[raw] += 1

                channel_hits = [c for c in channel_hits if c in active_map]
                if not channel_hits:
                    continue

                title = build_title(comp, ev=ev, prefix_womens=sport_is_womens(league_keys[0]))
                tag_key = next((k for k in league_keys if k in SPORT_TAGS), None)
                tag = f"({SPORT_TAGS.get(tag_key, sport_label)})"

                for ch in channel_hits:
                    rows.append({
                        "col_label": col_label,
                        "date": event_day,
                        "time_str": hm,
                        "start_dt": sort_dt,
                        "sport": sport_label,
                        "league_key": tag_key or league_keys[0],
                        "tag": tag,
                        "channel": ch,
                        "channel_num": active_map.get(ch),
                        "title": title,
                        "fav_row": None,   # normal grid
                        "source": "scoreboard",
                    })

# TV guide backup rows. These supplement the ESPN scoreboard output and are
# especially useful for Golf Channel / featured ESPN windows that do not map cleanly
# from the scoreboard feeds.
rows.extend(build_tvguide_rows())
# Golf-specific schedule backup for network/streaming golf windows that AYWT misses.
rows.extend(build_smw_golf_rows())

# Favorite pro teams (ignore channel filters; force into special rows)
FAV_PRO_ROWS = []     # [(row_num, row_label)]
FAV_ROW_STYLES = {}   # (row_num, row_label) -> {"bg_color": ..., "font_color": ..., "name": ...}

for i, fav in enumerate(FAVORITE_PRO_TEAMS):
    fav_cfg = _as_favorite_config(fav)
    team_name = fav_cfg["name"]
    league_keys = fav_cfg["leagues"]

    row_num = max(active_map.values()) + 10 + i     # place after normal channels in printed row list
    row_label = fav_cfg["row_label"]

    FAV_PRO_ROWS.append((row_num, row_label))
    FAV_ROW_STYLES[(row_num, row_label)] = {
        "name": team_name,
        "bg_color": fav_cfg["bg_color"],
        "font_color": fav_cfg["font_color"],
    }

    for day in DAY_DATES:
        events = fetch_day_sport_multi(day, league_keys)
        if not events:
            continue
        for ev in events:
            for comp in (ev.get("competitions") or []):
                if not _comp_has_team(comp, team_name):
                    continue
                local_dt, hm = to_local_timestr(comp.get("date") or ev.get("date"))
                if local_dt is None or local_dt.date() not in DAY_DATES:
                    continue
                col_label = _col_label(local_dt.date())
                title = build_title(comp, ev=ev, prefix_womens=sport_is_womens(league_keys[0]))
                tag = get_event_tag(next((k for k in league_keys if k in SPORT_TAGS), league_keys[0]), sport_label=team_name)
                rows.append({
                    "col_label": col_label,
                    "date": local_dt.date(),
                    "time_str": hm,
                    "start_dt": local_dt,
                    "sport": team_name,           # display purpose
                    "league_key": league_keys[0], # best-effort
                    "tag": tag,
                    "channel": row_label,
                    "channel_num": row_num,
                    "title": title,
                    "fav_row": team_name,         # mark favorite row by team name
                    "source": "favorite",
                })

# Favorite school (all listed college leagues; special styling)
SCHOOL_ROW_NUM   = 100 # sort after normal channels
SCHOOL_ROW_LABEL = FAVORITE_SCHOOL.get("row_label", FAVORITE_SCHOOL["name"])

FAV_ROW_STYLES[(SCHOOL_ROW_NUM, SCHOOL_ROW_LABEL)] = {
    "name": FAVORITE_SCHOOL["name"],
    "bg_color": FAVORITE_SCHOOL["bg_color"],
    "font_color": FAVORITE_SCHOOL["font_color"],
}

for day in DAY_DATES:
    for k in FAVORITE_SCHOOL["leagues"]:
        events = fetch_day_sport_multi(day, [k])
        if not events:
            continue
        for ev in events:
            for comp in (ev.get("competitions") or []):
                if not _comp_has_team(comp, FAVORITE_SCHOOL["name"]):
                    continue
                local_dt, hm = to_local_timestr(comp.get("date") or ev.get("date"))
                if local_dt is None or local_dt.date() not in DAY_DATES:
                    continue
                col_label = _col_label(local_dt.date())
                title = build_title(comp, ev=ev, prefix_womens=sport_is_womens(k))
                tag = f"({SPORT_TAGS.get(k, FAVORITE_SCHOOL['name'])})"
                rows.append({
                    "col_label": col_label,
                    "date": local_dt.date(),
                    "time_str": hm,
                    "start_dt": local_dt,
                    "sport": FAVORITE_SCHOOL["name"],
                    "league_key": k,
                    "tag": tag,
                    "channel": SCHOOL_ROW_LABEL,
                    "channel_num": SCHOOL_ROW_NUM,
                    "title": title,
                    "fav_row": FAVORITE_SCHOOL["name"],  # mark favorite row by team name
                    "source": "favorite",
                })


# ---------------- DataFrame ----------------
df = pd.DataFrame(rows)
if df.empty:
    print("No events matched. Check dates, SPORTS, favorites, and mapping.")
else:
    if "source" not in df.columns:
        df["source"] = "scoreboard"

    # When TV-guide rows duplicate a scoreboard row, keep the scoreboard version
    # because it usually has the better team-vs-team title and official start time.
    # First pass: drop guide rows that are close to a scoreboard row on the same
    # channel/date/tag even if the guide and scoreboard differ by a few minutes
    # (common for baseball broadcasts: 7:30 guide window vs 7:40 first pitch).
    if USE_TV_GUIDE_BACKUP and TV_GUIDE_DROP_GUIDE_ROWS_NEAR_SCOREBOARD and {"source", "start_dt", "date", "channel_num", "tag"}.issubset(df.columns):
        scoreboard_df = df[df["source"].eq("scoreboard")].copy()
        supplemental_df = df[df["source"].isin(["tvguide", "golf_smw"])].copy()
        guide_drop_idx = []
        if not scoreboard_df.empty and not supplemental_df.empty:
            scoreboard_df["start_dt"] = pd.to_datetime(scoreboard_df["start_dt"])
            supplemental_df["start_dt"] = pd.to_datetime(supplemental_df["start_dt"])
            overlap_window = pd.Timedelta(minutes=int(TV_GUIDE_SCOREBOARD_OVERLAP_MINUTES))
            for idx, guide_row in supplemental_df.iterrows():
                candidates = scoreboard_df[
                    scoreboard_df["date"].eq(guide_row["date"])
                    & scoreboard_df["channel_num"].eq(guide_row["channel_num"])
                    & scoreboard_df["tag"].eq(guide_row["tag"])
                ]
                if candidates.empty:
                    continue
                if ((candidates["start_dt"] - guide_row["start_dt"]).abs() <= overlap_window).any():
                    guide_drop_idx.append(idx)
        if guide_drop_idx:
            df = df.drop(index=guide_drop_idx).copy()
            print(
                f"Dropped {len(guide_drop_idx)} supplemental guide rows that overlapped a scoreboard row "
                f"within {TV_GUIDE_SCOREBOARD_OVERLAP_MINUTES} minutes"
            )

    # Second pass: exact duplicate guard. Golf scoreboard rows are suppressed
    # above by default, so Golf Channel guide rows survive.
    source_priority = {"favorite": 0, "scoreboard": 1, "tvguide": 2, "golf_smw": 3}
    df["_source_priority"] = df["source"].map(source_priority).fillna(9)
    before_dedupe = len(df)
    df = (
        df.sort_values(["_source_priority", "date", "channel_num", "start_dt", "title"])
          .drop_duplicates(["date", "channel_num", "channel", "start_dt", "tag"], keep="first")
          .drop(columns=["_source_priority"])
          .sort_values(["date", "channel_num", "start_dt", "title"])
          .reset_index(drop=True)
    )
    dropped = before_dedupe - len(df)
    if dropped:
        print(f"Dropped {dropped} duplicate TV/scoreboard rows")

    df.to_csv(os.path.join(OUTPUT_DIR, "events_flat_storypoint.csv"), index=False)
    print("Wrote flat CSV with", len(df), "rows")

# ========================== EXCEL WRITER (rich text, borders, alt shading + fav rows) ==========================
if not df.empty:
    df["col_label"] = pd.Categorical(df["col_label"], categories=COL_LABELS, ordered=True)

    # Per-cell: (channel_num, channel, col_label) -> [(time, tag, title, fav_row_flag)]
    events_by_cell = defaultdict(list)
    for r in df.itertuples():
        events_by_cell[(r.channel_num, r.channel, r.col_label)].append((r.time_str, r.tag, r.title, r.fav_row))

    # Row order: favorites first, then normal channels by number
    active_map = CHANNEL_MAPS[ACTIVE_CHANNEL_MAP_NAME]
    normal_rows = sorted([(num, ch) for ch, num in active_map.items()], key=lambda x: (x[0], str(x[1])))
    fav_rows = [(SCHOOL_ROW_NUM, SCHOOL_ROW_LABEL)] + FAV_PRO_ROWS
    row_index = fav_rows + normal_rows

    # title/subheader
    def fmt(d: date) -> str: return d.strftime("%B %d, %Y")
    TITLE_TEXT   = f"{fmt(START_DATE)} – {fmt(DAY_DATES[-1])}"
    SUBHEAD_TEXT = f"StoryPoint – East Lansing | Created {datetime.now(LOCAL_TZ).strftime('%B %d, %Y')} by J.Smith"

    # column positions
    first_col     = 1
    num_col       = first_col
    chan_col      = first_col + 1
    first_day_col = chan_col + 1
    last_day_col  = chan_col + len(COL_LABELS)

    out_path = os.path.join(OUTPUT_DIR, OUTPUT_XLSX)
    with pd.ExcelWriter(out_path, engine="xlsxwriter") as xw:
        wb  = xw.book
        ws  = wb.add_worksheet("Week")

        # Per-sport font colors
        SPORT_STYLE = {
            "NFL": "#B22222", "NBA": "#5C2E91", "NHL": "#1F4E79", "MLB": "#0A2463",
            "SOFTBALL": "#BE123C", "Softball": "#BE123C",
            "FBS FB": "#0B6E4F", "CFB": "#0B6E4F", "M CBB": "#D97706", "W CBB": "#C026D3",
            "MBB": "#D97706", "WBB": "#C026D3",
            "MLS": "#0B7285", "EPL": "#1D4ED8", "UCL": "#6D28D9", "MCH": "#065F46",
            "SOCCER - MLS": "#0B7285", "SOCCER - EPL": "#1D4ED8", "SOCCER - UCL": "#6D28D9",
            "PGA": "#047857", "PGA TOUR": "#047857", "LPGA": "#BE185D", "LIV": "#111827", "DPW": "#B45309", "KFT": "#374151",
            "GOLF": "#047857", "COLLEGE GOLF": "#047857", "CHAMPIONS TOUR": "#065F46",
            "UFL": "#7C2D12", "RUGBY": "#7C2D12", "FIGHTING": "#991B1B", "HORSE RACING": "#854D0E",
            "NCAA BASEBALL": "#92400E", "WNBA": "#C026D3", "SOCCER": "#0B7285", "TENNIS": "#166534",
            "RACING": "#7C2D12", "VOLLEYBALL": "#9333EA", "TV": "#374151",
        }

        # Base formats
        title_fmt = wb.add_format({"bold": True, "align": "left", "valign": "vcenter", "font_size": 36 })
        sub_fmt   = wb.add_format({"italic": True, "align": "left", "valign": "vcenter", "font_size": 12, "font_color": "#555555"})

        header_fmt = wb.add_format({"bold": True, "align": "center", "valign": "vcenter", "font_size": 14, "border": 1})

        num_fmt         = wb.add_format({"bold": True, "align": "center", "valign": "vcenter", "font_size": 24, "bottom": 1})
        chan_fmt        = wb.add_format({"bold": True, "align": "center",  "valign": "vcenter", "font_size": 22, "bottom": 1, "text_wrap": True})
        num_fmt_shaded  = wb.add_format({"bold": True, "align": "center", "valign": "vcenter", "font_size": 24, "bottom": 1, "bg_color": "#F5F5F5"})
        chan_fmt_shaded = wb.add_format({"bold": True, "align": "center",  "valign": "vcenter", "font_size": 22, "bottom": 1, "bg_color": "#F5F5F5", "text_wrap": True})

        base_cell_fmt        = wb.add_format({"text_wrap": True, "valign": "vcenter", "bottom": 1})
        base_cell_fmt_shaded = wb.add_format({"text_wrap": True, "valign": "vcenter", "bottom": 1, "bg_color": "#F5F5F5"})

        # Favorite-row formats are generated from FAV_ROW_STYLES so every favorite
        # can use its own team colors, not just MSU.
        _favorite_format_cache = {}

        def _get_favorite_formats(fav_style: dict):
            bg = fav_style.get("bg_color", "#F5F5F5")
            fg = fav_style.get("font_color", "#000000")
            key = (bg, fg)
            if key in _favorite_format_cache:
                return _favorite_format_cache[key]

            cell_fmt = wb.add_format({
                "text_wrap": True,
                "valign": "vcenter",
                "bottom": 1,
                "bg_color": bg,
                "font_color": fg,
            })
            time_fmt = wb.add_format({
                "bold": True,
                "font_size": 16,
                "font_color": fg,
            })
            teams_fmt = wb.add_format({
                "font_size": 14,
                "font_color": fg,
            })
            num_hdr_fmt = wb.add_format({
                "bold": True,
                "align": "center",
                "valign": "vcenter",
                "font_size": 24,
                "bottom": 1,
                "bg_color": bg,
                "font_color": fg,
            })
            chan_hdr_fmt = wb.add_format({
                "bold": True,
                "align": "center",
                "valign": "vcenter",
                "font_size": 22,
                "bottom": 1,
                "bg_color": bg,
                "font_color": fg,
                "text_wrap": True,
            })

            _favorite_format_cache[key] = (cell_fmt, time_fmt, teams_fmt, num_hdr_fmt, chan_hdr_fmt)
            return _favorite_format_cache[key]

        # Titles
        ws.merge_range(0, num_col, 0, last_day_col, f"Live Sports on TV - Week of {fmt(START_DATE)}", title_fmt)
        ws.merge_range(1, num_col, 1, last_day_col, SUBHEAD_TEXT, sub_fmt)

        # Column headers
        ws.write(2, num_col,  "#", num_fmt)
        ws.write(2, chan_col, "Channel", chan_fmt)
        for c, lbl in enumerate(COL_LABELS):
            ws.write(2, first_day_col + c, lbl, header_fmt)

        # Widths
        ws.set_column(num_col,  num_col, 12)
        ws.set_column(chan_col, chan_col, 20)   # wider for "Streaming – …"
        ws.set_column(first_day_col, last_day_col, 36)

        # Helpers
        def _tag_key(tag_text: str) -> str:
            return (tag_text or "").strip().strip("()").upper()

        _format_cache = {}
        def _get_event_formats(tag: str, shaded: bool):
            key = (tag, shaded)
            if key in _format_cache:
                return _format_cache[key]
            color = SPORT_STYLE.get(tag)
            time_kwargs = {"bold": True, "font_size": 16}
            team_kwargs = {"font_size": 14}
            if color:
                time_kwargs["font_color"] = color
                team_kwargs["font_color"] = color
            tt_fmt = wb.add_format(time_kwargs)
            tm_fmt = wb.add_format(team_kwargs)
            cell_fmt = base_cell_fmt_shaded if shaded else base_cell_fmt
            _format_cache[key] = (tt_fmt, tm_fmt, cell_fmt)
            return _format_cache[key]

        def write_rich_cell(row_idx: int, col_idx: int, evts, shaded: bool, fav_style: dict | None):
            """
            evts: list of (time_str, tag_text, title, fav_row_flag)
            fav_style: None for normal rows, or {"bg_color": ..., "font_color": ...}
            """
            if fav_style:
                fav_cell_fmt, fav_time_fmt, fav_teams_fmt, _, _ = _get_favorite_formats(fav_style)
            else:
                fav_cell_fmt = fav_time_fmt = fav_teams_fmt = None

            if not evts:
                # choose correct base format
                base_fmt = (
                    fav_cell_fmt if fav_style
                    else (base_cell_fmt_shaded if shaded else base_cell_fmt)
                )
                ws.write(row_idx, col_idx, "", base_fmt)
                return

            parts = []
            for i, (time_str, tag_text, title, _) in enumerate(evts):
                tag = _tag_key(tag_text)
                if fav_style:
                    # use this favorite team's row colors
                    parts.extend([fav_time_fmt, f"{time_str}: ({tag})"])
                    parts.append("\n")
                    parts.extend([fav_teams_fmt, _format_title_for_grid_cell(title, tag)])
                else:
                    tt_fmt, tm_fmt, _ = _get_event_formats(tag, shaded)
                    parts.extend([tt_fmt, f"{time_str}: ({tag})"])
                    parts.append("\n")
                    parts.extend([tm_fmt, _format_title_for_grid_cell(title, tag)])
                if i != len(evts) - 1:
                    parts.append("\n")

            # trailing format controls background/wrap
            trailing_fmt = (
                fav_cell_fmt if fav_style
                else (base_cell_fmt_shaded if shaded else base_cell_fmt)
            )
            parts.append(trailing_fmt)
            ws.write_rich_string(row_idx, col_idx, *parts)

        # Body with alternating shading for normal rows; favorite rows use team colors
        current_row = 3
        for idx, (num, ch) in enumerate(row_index):
            fav_style = FAV_ROW_STYLES.get((num, ch))
            if fav_style:
                shaded = False     # favorite rows have their own bg
                _, _, _, row_num_fmt, row_chan_fmt = _get_favorite_formats(fav_style)
            else:
                shaded = (idx % 2 == 1)
                row_num_fmt  = num_fmt_shaded  if shaded else num_fmt
                row_chan_fmt = chan_fmt_shaded if shaded else chan_fmt

            # Row header cells
            try:
                ws.write(current_row, num_col, int(num) if num is not None else "", row_num_fmt)
            except Exception:
                ws.write(current_row, num_col, "", row_num_fmt)
            ws.write(current_row, chan_col, str(ch), row_chan_fmt)

            # Day cells
            for c, lbl in enumerate(COL_LABELS):
                evts = events_by_cell.get((num, ch, lbl), [])
                write_rich_cell(current_row, first_day_col + c, evts, shaded, fav_style)

            current_row += 1

                # Footer or summary rows could be added here if needed

        # ---- Page setup / print settings ----
        last_row = current_row - 1              # last row we wrote
        first_row = 0                           # include title/subheader
        first_col = num_col                     # start at the "#" column
        last_col  = last_day_col                # last day column

        # Define the print area so the sheet opens ready to print.
        ws.print_area(first_row, first_col, last_row, last_col)

        # Repeat the date/header row on printed pages.
        ws.repeat_rows(2, 2)  # zero-based row index; headers are on row 2

        # Paper and margins.
        ws.set_paper(1)       # 1 = Letter (8.5" x 11"). Use 9 for A4.
        ws.set_margins(left=0.2, right=0.2, top=0.2, bottom=0.2)

        # For 1–4 days, keep the original compact one-page landscape handout.
        # For 5+ days, split the days into print pages of at most 4 day columns.
        # The # and Channel columns repeat on the second page, which makes a
        # Monday–Sunday run work cleanly as a duplex/front-and-back print.
        max_days_per_print_page = 4
        day_count = len(COL_LABELS)
        if day_count > max_days_per_print_page:
            pages_wide = (day_count + max_days_per_print_page - 1) // max_days_per_print_page
            ws.set_portrait()
            ws.fit_to_pages(pages_wide, 1)
            ws.repeat_columns(num_col, chan_col)

            # Insert vertical page breaks before day 5, day 9, etc.
            vertical_breaks = [
                first_day_col + i
                for i in range(max_days_per_print_page, day_count, max_days_per_print_page)
            ]
            if vertical_breaks:
                ws.set_v_pagebreaks(vertical_breaks)

            ws.center_horizontally()
        else:
            ws.set_landscape()
            ws.fit_to_pages(1, 1)
            ws.center_horizontally()
            ws.center_vertically()

        # Optional: show/hide gridlines in print (2 = hide on screen & print)
        ws.hide_gridlines(2)

        # Optional: header/footer
        ws.set_header('&L&"Calibri,Bold"&12StoryPoint – East Lansing'
                    '&R&"Calibri"&10Printed &D')
        # ws.set_footer('&CPage &P of &N')



    print("Wrote workbook:", out_path)

# ========================== UNKNOWN BROADCASTS LOG ==========================
if unknown_broadcasts:
    pd.DataFrame(
        sorted(unknown_broadcasts.items(), key=lambda x: (-x[1], x[0])),
        columns=["raw_broadcast_string","count"]
    ).to_csv(os.path.join(OUTPUT_DIR, "unknown_broadcasts.csv"), index=False)
    print("Wrote unknown_broadcasts.csv (consider adding aliases).")


TV guide ESPN: parsed 111 candidate station listings
TV guide ESPN2: parsed 75 candidate station listings
TV guide ESPNU: parsed 19 candidate station listings
TV guide ESPNews: parsed 58 candidate station listings
TV guide Golf: parsed 24 candidate station listings
TV guide FS1: parsed 15 candidate station listings
TV guide TNT: parsed 3 candidate station listings
TV guide TBS: parsed 1 candidate station listings
TV guide USA: parsed 3 candidate station listings
TV guide NBC: parsed 1 candidate station listings
TV guide CBS: parsed 2 candidate station listings
TV guide FOX: parsed 1 candidate station listings
TV guide ABC: parsed 2 candidate station listings
TV guide backup parsed 315 total listings; kept 36 candidate event rows; rejected 279
TV guide kept rows by channel/tag:
channel              tag  rows
    ABC           (WNBA)     2
    CBS       (PGA Tour)     2
   ESPN            (NHL)     1
    FOX            (MLB)     1
    FS1            (MLB)     2
    FS1         (Racing)  

In [2]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)